# Camada Silver — Consolidação de Limpeza e Integridade Referencial

Consolida os logs `*_silver_log.json` das 8 tabelas tratadas, valida a integridade referencial de `id_aluno` em Silver e gera `resumo_silver.csv` e `relatorio_limpeza_silver.json`.

In [1]:
import json
import sys
from pathlib import Path

import pandas as pd

for candidate in [Path.cwd(), Path.cwd().parent, Path.cwd().parent.parent]:
    if (candidate / "notebooks" / "utils").exists():
        if str(candidate / "notebooks") not in sys.path:
            sys.path.insert(0, str(candidate / "notebooks"))
        break

from utils.silver_utils import get_silver_paths

_, _, SILVER_DIR = get_silver_paths()

alunos_ids = set(pd.read_parquet(SILVER_DIR / "alunos.parquet", columns=["id_aluno"])["id_aluno"].dropna())
tables = ["alunos", "matriculas", "notas", "acessos", "redes", "financeiro", "deslocamento", "frequencia"]

relatorio = {}
rows = []
for name in tables:
    with open(SILVER_DIR / f"{name}_silver_log.json", "r", encoding="utf-8") as f:
        log = json.load(f)
    ids = set(pd.read_parquet(SILVER_DIR / f"{name}.parquet", columns=["id_aluno"])["id_aluno"].dropna())
    match_pct = round(100.0 * len(ids & alunos_ids) / len(ids), 2) if ids else 0.0
    log["id_aluno_match_alunos_pct"] = match_pct
    relatorio[name] = log
    rows.append({
        "tabela": name,
        "linhas_bronze": log["rows_in"],
        "linhas_silver": log["rows_out"],
        "duplicatas_removidas": log["duplicates_removed"],
        "id_aluno_unicos": len(ids),
        "match_alunos_pct": match_pct,
    })

resumo = pd.DataFrame(rows)
resumo.to_csv(SILVER_DIR / "resumo_silver.csv", index=False, encoding="utf-8-sig")
with open(SILVER_DIR / "relatorio_limpeza_silver.json", "w", encoding="utf-8") as f:
    json.dump(relatorio, f, ensure_ascii=False, indent=2)

print(resumo.to_string(index=False))

      tabela  linhas_bronze  linhas_silver  duplicatas_removidas  id_aluno_unicos  match_alunos_pct
      alunos        1000000        1000000                     0          1000000             100.0
  matriculas         700000         700000                     0           376925             100.0
       notas         600000         600000                     0           349170             100.0
     acessos         400000         400000                     0           275613             100.0
       redes         300000         300000                     0           225241             100.0
  financeiro        1000000         993804                  4983            14001             100.0
deslocamento           1000           1000                     0              996             100.0
  frequencia             20             20                     0                9             100.0
